<a href="https://colab.research.google.com/github/Tarunvaka/ps1/blob/main/ps1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ps1 - I/O + descriptive stats
**Tarun Vaka** ·
Quick global-health snapshot: is national income tied to how long people live, and what does the US population look like inside those countries? Three tiny datasets (~5 vars, ~50 obs each) loaded from three different formats.

**Data**
1. **HTML** - Wikipedia, *List of countries by life expectancy* - `https://en.wikipedia.org/wiki/List_of_countries_by_life_expectancy`
2. **JSON** - World Bank Indicators API, GDP per capita (2022) - `https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD?format=json&date=2022&per_page=400`
3. **SAS xpt** - CDC NHANES 2021-2023 Demographics - `https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.xpt`


In [ ]:
import io, json, urllib.request
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 12)

UA = {"User-Agent": "Mozilla/5.0 (ps1 educational)"}   # Wikipedia blocks default urllib UA


## 1. HTML - life expectancy from Wikipedia

In [ ]:
URL_LE = "https://en.wikipedia.org/wiki/List_of_countries_by_life_expectancy"

req = urllib.request.Request(URL_LE, headers=UA)
with urllib.request.urlopen(req) as r:
    html = r.read().decode("utf-8")

tables = pd.read_html(io.StringIO(html))
print(f"{len(tables)} tables found; shapes: {[t.shape for t in tables[:5]]}")


12 tables found; shapes: [(1, 1), (211, 21), (210, 16), (206, 20), (185, 22)]


In [ ]:
# Pick the first table that mentions "life expectancy" in its columns.
def is_life_table(t):
    flat = [" ".join(map(str, c)) if isinstance(c, tuple) else str(c) for c in t.columns]
    return any("life expectancy" in s.lower() for s in flat)

le = next(t for t in tables if is_life_table(t)).copy()

# Flatten multi-index columns.
le.columns = [" ".join(map(str, c)).strip().lower() if isinstance(c, tuple) else str(c).strip().lower()
              for c in le.columns]
le.columns[:8]


Index(['locations locations', 'life expectancy overall at birth', 'life expectancy overall bonus 0→15',
       'life expectancy overall at 15', 'life expectancy overall bonus 15→65', 'life expectancy overall at 65',
       'life expectancy overall bonus 65→80', 'life expectancy overall at 80'],
      dtype='object')

In [ ]:
# Keep 5 useful cols: country + all-sex / male / female life expectancy + rank if present.
def pick(cols, *needles):
    for n in needles:
        for c in cols:
            if all(w in c for w in n.split()):
                return c
    return None

col_country = pick(le.columns, "country", "location", "state")
col_both    = pick(le.columns, "all life", "both life", "overall life expectancy", "all-sex life", "life expectancy all", "life expectancy overall", "life expectancy both", "life expectancy")
col_male    = pick(le.columns, "male life") or next((c for c in le.columns if "male" in c and "female" not in c), None)
col_female  = pick(le.columns, "female life") or next((c for c in le.columns if "female" in c), None)

keep = {"country": col_country, "life_both": col_both, "life_male": col_male, "life_female": col_female}
keep = {k: v for k, v in keep.items() if v}
le = le[list(keep.values())].rename(columns={v: k for k, v in keep.items()})

for c in le.columns:
    if c != "country":
        le[c] = pd.to_numeric(le[c], errors="coerce")

le = le.dropna(subset=["life_both"]).reset_index(drop=True)
le = le.head(50)   # keep a teaching-sized frame
le.shape, le.head()


((50, 4),
             country  life_both  life_male  life_female
 0         Hong Kong      85.51      82.84        88.13
 1             Japan      84.71      81.69        87.74
 2       South Korea      84.33      81.19        87.16
 3  French Polynesia      84.07      81.78        86.50
 4           Andorra      84.04      82.10        86.11)

In [ ]:
# Descriptive stats
le.describe().round(1)


,life_both,life_male,life_female
count,50.0,50.0,50.0
mean,82.4,80.2,84.6
std,1.3,1.3,1.4
min,80.0,77.6,81.8
25%,81.6,79.2,83.7
50%,82.3,80.4,84.6
75%,83.3,81.2,85.7
max,85.5,82.8,88.1


In [ ]:
# Subset on condition: which of the top 50 rows have life expectancy above 82?
long_lived = le.loc[le["life_both"] > 82, ["country", "life_both"]].sort_values("life_both", ascending=False)
long_lived


,country,life_both
0,Hong Kong,85.51
1,Japan,84.71
2,South Korea,84.33
3,French Polynesia,84.07
4,Andorra,84.04
5,Switzerland,83.95
6,Australia,83.92
7,Singapore,83.74
8,Italy,83.72
9,Spain,83.67


In [ ]:
# Regex filter: countries whose name contains "Republic", "stan", or hyphen — a quick sanity look at naming patterns
le[le["country"].str.contains(r"Republic|stan|-", case=False, regex=True)][["country", "life_both"]]


,country,life_both


In [ ]:
# Groupby/agg: bucket the 50-row slice into life-expectancy quartiles
le["le_bucket"] = pd.qcut(le["life_both"], 4, labels=["Q1_lowest", "Q2", "Q3", "Q4_highest"])
le.groupby("le_bucket", observed=True)["life_both"].agg(["count", "mean", "min", "max"]).round(2)


,count,mean,min,max
le_bucket,,,,
Q1_lowest,13,80.85,80.03,81.60
Q2,12,82.00,81.65,82.31
Q3,12,82.77,82.36,83.30
Q4_highest,13,83.99,83.31,85.51


_Interpretation:_ mean life expectancy in this 50-row slice sits in the low-80s (the table is sorted best-first). The `life_both > 82` subset picks out the usual suspects - Japan, Switzerland, small European states - confirming the table parsed correctly.

## 2. JSON - GDP per capita from the World Bank API

In [ ]:
URL_GDP = ("https://api.worldbank.org/v2/country/all/indicator/"
           "NY.GDP.PCAP.CD?format=json&date=2022&per_page=400")

with urllib.request.urlopen(URL_GDP) as r:
    payload = json.load(r)

records = payload[1]                          # World Bank returns [metadata, records]
gdp = pd.json_normalize(records)
gdp = (gdp[["country.value", "countryiso3code", "value"]]
       .rename(columns={"country.value": "country",
                        "countryiso3code": "iso3",
                        "value": "gdp_per_capita_usd"}))
gdp = gdp.dropna(subset=["gdp_per_capita_usd", "iso3"])
gdp = gdp[gdp["iso3"].str.len() == 3]         # drop regional aggregates
gdp["log_gdp_pc"] = np.log10(gdp["gdp_per_capita_usd"])
gdp = gdp.sort_values("gdp_per_capita_usd", ascending=False).reset_index(drop=True)
gdp.head()


,country,iso3,gdp_per_capita_usd,log_gdp_pc
0,Monaco,MCO,226052.060665,5.354208
1,Liechtenstein,LIE,188055.003235,5.274285
2,Luxembourg,LUX,123719.658916,5.092439
3,Bermuda,BMU,119969.173269,5.079070
4,Norway,NOR,113122.130766,5.053548


In [ ]:
gdp[["gdp_per_capita_usd", "log_gdp_pc"]].describe().round(2)


,gdp_per_capita_usd,log_gdp_pc
count,252.00,252.00
mean,19671.18,3.90
std,29259.75,0.61
min,301.83,2.48
25%,2679.16,3.43
50%,7630.92,3.88
75%,24450.04,4.39
max,226052.06,5.35


In [ ]:
# Regexp filter: every country whose English name ends in "stan"
stan = gdp[gdp["country"].str.contains(r"stan$", case=False, regex=True)]
stan[["country", "gdp_per_capita_usd"]]


,country,gdp_per_capita_usd
103,Kazakhstan,11255.339644
140,"Middle East, North Africa, Afghanistan & Pakistan",6436.740303
153,Turkmenistan,5198.189474
188,Uzbekistan,2698.602630
217,Pakistan,1538.322813
230,Tajikistan,1052.179495
250,Afghanistan,357.261153


In [ ]:
# Groupby/agg: quartile-bucket the world by GDP per capita
gdp["gdp_bucket"] = pd.qcut(gdp["gdp_per_capita_usd"], 4, labels=["Q1_low", "Q2", "Q3", "Q4_high"])
gdp.groupby("gdp_bucket", observed=True)["gdp_per_capita_usd"].agg(["count", "mean", "median"]).round(0)


,count,mean,median
gdp_bucket,,,
Q1_low,63,1453.0,1417.0
Q2,63,4926.0,4754.0
Q3,63,14103.0,13220.0
Q4_high,63,58202.0,48825.0


_Interpretation:_ the GDP distribution is heavily right-skewed — the Q4 mean is roughly an order of magnitude above the Q1 mean - which is why the log column matters when we merge with life expectancy below.

## 3. SAS (.xpt) - NHANES 2021-2023 demographics

In [ ]:
URL_DEMO = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.xpt"

# pandas.read_sas reads .xpt (XPORT) format straight from the URL.
demo_full = pd.read_sas(URL_DEMO, format="xport")
print(demo_full.shape)

demo = (demo_full[["SEQN", "RIAGENDR", "RIDAGEYR", "RIDRETH3", "INDFMPIR"]]
        .rename(columns={"SEQN": "id", "RIAGENDR": "gender", "RIDAGEYR": "age",
                         "RIDRETH3": "race", "INDFMPIR": "poverty_ratio"})
        .dropna(subset=["age"]))

demo["gender"] = demo["gender"].map({1: "Male", 2: "Female"})
demo["race"]   = demo["race"].map({1: "Mex-American", 2: "Other Hispanic",
                                   3: "NH-White",     4: "NH-Black",
                                   6: "NH-Asian",     7: "Other/Mixed"})

demo = demo.sample(50, random_state=42).reset_index(drop=True)   # 50-row teaching slice
demo.head()


(11933, 27)


,id,gender,age,race,poverty_ratio
0,140550.0,Male,63.0,NH-White,5.00
1,140700.0,Male,11.0,Other/Mixed,NaN
2,139573.0,Female,13.0,Mex-American,NaN
3,130735.0,Female,10.0,Mex-American,NaN
4,134730.0,Female,26.0,NH-White,2.73


In [ ]:
demo[["age", "poverty_ratio"]].describe().round(2)


,age,poverty_ratio
count,50.00,38.00
mean,31.74,2.72
std,24.06,1.73
min,2.00,0.00
25%,11.00,1.38
50%,26.50,2.22
75%,49.75,5.00
max,80.00,5.00


In [ ]:
# Subset on condition: working-age adults (18-64) with below-poverty household income.
below_poverty_adults = demo.query("18 <= age <= 64 and poverty_ratio < 1.0")
below_poverty_adults[["id", "age", "gender", "race", "poverty_ratio"]]


,id,age,gender,race,poverty_ratio
18,140971.0,43.0,Female,NH-White,5.397605e-79
27,132758.0,33.0,Female,NH-Black,8.400000e-01
28,130774.0,50.0,Female,NH-White,2.300000e-01
32,137009.0,56.0,Female,NH-Black,1.600000e-01


In [ ]:
# Regexp filter: any race label containing "Hispanic" (case-insensitive).
hispanic = demo[demo["race"].fillna("").str.contains(r"hispanic", case=False, regex=True)]
hispanic["race"].value_counts()


race
Other Hispanic    4
Name: count, dtype: int64

In [ ]:
# Groupby/agg: mean age and mean poverty ratio by race x gender.
demo.groupby(["race", "gender"], observed=True).agg(
    n=("id", "count"),
    mean_age=("age", "mean"),
    mean_pir=("poverty_ratio", "mean"),
).round(2)


n  mean_age  mean_pir
race           gender                        
Mex-American   Female   3     10.33       NaN
               Male     1     51.00      1.95
NH-Asian       Female   2     26.50      3.16
               Male     2     44.00      3.58
NH-Black       Female   6     42.17      2.17
               Male     4     17.75      1.84
NH-White       Female  13     27.62      2.24
               Male    10     45.30      3.53
Other Hispanic Female   2     13.50      3.01
               Male     2     23.50      2.26
Other/Mixed    Male     5     30.80      2.92

_Interpretation:_ even in a 50-person random slice, the contrasts NHANES is designed to expose show up - race×gender cells have visibly different mean ages and poverty ratios. A real analysis would use survey weights (`WTINT2YR`); this problem set is about I/O, not inference.

## 4. Bonus join - life expectancy vs GDP per capita

In [ ]:
merged = le.merge(gdp[["country", "gdp_per_capita_usd", "log_gdp_pc"]], on="country", how="inner")
merged.shape


(41, 7)

In [ ]:
merged[["life_both", "gdp_per_capita_usd", "log_gdp_pc"]].corr().round(3)


,life_both,gdp_per_capita_usd,log_gdp_pc
life_both,1.000,0.128,0.184
gdp_per_capita_usd,0.128,1.000,0.949
log_gdp_pc,0.184,0.949,1.000


In [ ]:
merged.sort_values("life_both", ascending=False).head(5)[["country", "life_both", "gdp_per_capita_usd"]]


,country,life_both,gdp_per_capita_usd
0,Japan,84.71,35548.264522
1,French Polynesia,84.07,20053.956937
2,Andorra,84.04,42414.059011
3,Switzerland,83.95,97809.095567
4,Australia,83.92,65169.519112


_So what:_ within the top-50 life-expectancy slice, the raw Pearson correlation with GDP per capita is only ~0.13 (0.18 on log GDP). That is the truncation-of-range story - once you condition on being a long-lived country, income no longer sorts you very hard. The classic Preston curve steepens on the low-income end that this slice deliberately drops. The NHANES peek reminds us that even inside a single rich country the household-income spread is wide enough to matter for outcomes.